In [1]:
import sys
import os

# Dodanie katalogu głównego repo do ścieżki
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from pipeline.parametres_search import scan_turing_am, a_m_pairs
from pipeline.pattern_visualization import simulate_patterns, plot_patterns, save_as_npz, define_patterns, convert_to_csv

# Szybka teoria
Zakres parametrów:

* $a \in (0, \infty)$
* $m \in (0, \frac{a}{2})$
* $d_1 \in (d_2, \infty)$
* $d_2 \in (0, \infty)$

Założenia, z których korzystamy:

- $a^2 - 4m^2 \geq 0$ z $\Delta$ stanu stacjonarnego
- $d_1 \geq d_2$

# Test
W teście skorzystamy z przewidywania istnienia Turinga, więc musimy jedynie dokładnie przetestować $d_1$ i $d_2$

In [1]:
# FIZYCZNE PARAMETRY
from pipeline.model_core import dimensional_to_dimensionless
A = [250, 500] # zakres; jednostkka kg * H20 * m^(-2) * rok^(-1)
L = 4 # rok^(-1)
J_tree = 0.002 # (kg * H20)^(-1)
J_grass = 0.003 # (kg * H20)^(-1)
M_tree = 0.18 # rok^(-1)
M_grass = 1.8 # rok^(-1)

# Brakuje: R, DW, DN
# R - pobor wody, czym większy, tym lepiej (zwiększa nam a)
# min dla drzew 2,0736
# min dla trawy 92,16
R_tree = 3 # kg * H20 * m^(-2) * rok^(-1) (kg dry mass)^(-2)
R_grass = 100 # kg * H20 * m^(-2) * rok^(-1) (kg dry mass)^(-2)

# Dw - dyfuzja wody (własne)
Dw = 10**5

# Dn - dyfuzja biomasy (własne)
Dn = 10**2

# Lx z ustaleń
Lx = 20

tryby = ["drzewa", "grass"]
tryb = tryby[1]

if tryb == "drzewa":
    J = J_tree
    M = M_tree
    R = R_tree
else:
    J = J_grass
    M = M_grass
    R = R_grass

a1, m, d1, d2 = dimensional_to_dimensionless(A[0], L, R, Dw, J, M, Dn, Lx)
a2, _, _, _ = dimensional_to_dimensionless(A[1], L, R, Dw, J, M, Dn, Lx)

print([a1, a2], m, d1, d2)

[np.float64(0.9374999999999999), np.float64(1.8749999999999998)] 0.45 62.5 0.0625


## Badanie parametru $d_1$
stałe $d_2 = 0.02$ i $d_1 \in [1,100]$

In [2]:
d2 = 0.02
d1_vals = np.arange(1, 100, 1) # można zmienic co ile

## Badanie parametru $d_2$
stałe $d_1 = 100$ i $d_2 \in [0,1]$

In [3]:
d1 = 100
d2_vals = np.arange(0.01, 1, 0.01) # można zmienic co ile i odkąd o ile >0

## Jak generować?
Parametr $d_1$

In [ ]:
# parametry stałe wszędzie (nie tykać!!!!)
Lx = 20
Ly = Lx
Nx = 100
Ny = Nx
mvals = np.arange(0.1, 1.51, 0.1)
avals = np.linspace(0.0, 5.0, 250)

# Parametr stały dla eksperymentu
d2 = 0.02

# podaj parametry
#d1_vals = np.arange(1, 100, 5) # np.arange(1, 100, co ile)
d1_vals = [5]

# Pętla
all_results = {}
a = []
m = []
d1v = []

for d1 in d1_vals:
    results = scan_turing_am(d1, d2, m_values=mvals, a_values=avals, k_min=0, k_max=20, n_k=1000)

    all_results[(d1, d2)] = results

    chosen = a_m_pairs(results, mvals)

    for row in chosen:
        m.extend([row["m"], row["m"]])
#        a.extend([row["a_max"], row["a_mean"]])
        a.extend([row["a_max"], row["a_median"]])
        d1v.extend([d1, d1])

d1_list = d1v
d2_list = [d2] * len(m)

if len(a) == len(m) == len(d1_list) == len(d2_list):
    print("ok")
else:
    raise ValueError("Wektory mają różne długości")

save_as_npz("plik", a, m, d1_list, d2_list)
define_patterns("plik")
convert_to_csv("plik")

Parametr $d_2$

In [4]:
import sys
import os

# Dodanie katalogu głównego repo do ścieżki
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from pipeline.parametres_search import scan_turing_am, a_m_pairs
from pipeline.pattern_visualization import simulate_patterns, plot_patterns, save_as_npz, define_patterns, convert_to_csv

# parametry stałe wszędzie (nie tykać!!!!)
Lx = 60
Ly = Lx
Nx = 100
Ny = Nx
#mvals = np.arange(0.1, 1.51, 0.1)
mvals = [0.45]
avals = np.linspace(0.0, 5.0, 250)

# Parametr stały dla eksperymentu
d1 = 100

# podaj parametry
#d2_vals = np.arange(0.1, 1, 0.01) # np.arrange(>0, 1, co ile)
d2_vals = [0.02]

# Pętla
all_results = {}
a, m, d2v = [], [], []

for d2 in d2_vals:
    results = scan_turing_am(d1, d2,m_values=mvals,a_values=avals,k_min=0, k_max=20, n_k=8000)

    all_results[(d1, d2)] = results
    chosen_ones = a_m_pairs(results, mvals)

    for row in chosen_ones:
        for aval in row["a_band"]:
            a.append(aval)
            m.append(row["m"])
            d2v.append(d2)
d2_list = d2v
d1_list = [d1] * len(m)

if len(a) == len(m) == len(d1_list) == len(d2_list):
    print("ok")
else:
    raise ValueError("Wektory mają różne długości")

print(len(a))


ok
5


In [2]:
save_as_npz("plik2", a, m, d1_list, d2_list)
#convert_to_csv("plik")
define_patterns("plik2")

Koniec zapisu.
Koniec. Etykiety ma 0/0 macierzy.
Plik zapisano do: wykresy_etykiety\plik2.npz


In [ ]:
dane = np.load("plik.npz")
atest = dane["a"]
mtest = dane["m"]
for i in range(len(mtest)):
    print(atest[i], mtest[i])

In order to obtain Turing patterns under laboratory conditions, it is required that diffusion constants of the two
species have a sufficiently large ratio. (Suzuki)
